# Fine-tune X3D-M on THETIS RGB (Kaggle)

End-to-end: download THETIS RGB from GitHub, validate it, fine-tune Kinetics-400 pretrained **X3D-M** with **player-grouped** splits, and report test accuracy.

**Kaggle settings before running:** Accelerator = GPU (T4 or P100), **Internet = On** (needed for `git clone` and the pretrained weights).

Notes
- Splits are by player, never by video, so no player appears in more than one of train/val/test.
- The best epoch is chosen on validation, and test accuracy is reported once at that epoch.
- No horizontal flips (they would turn forehands into backhands).

In [1]:
!pip -q install fvcore iopath
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
torch 2.11.0+cu130 | cuda: True | Tesla T4


## 1. Config

In [2]:
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
DATA_DIR = WORK / "thetis"
RGB_ROOT = DATA_DIR / "VIDEO_RGB"

# Optional: re-encode videos to a 256 short side first (saves a bit of decode time per epoch)
RESIZE = False
RESIZED_ROOT = WORK / "thetis_256"

# Training
FOLD = 0              # which of N_FOLDS player-grouped folds is the test fold
N_FOLDS = 5
RUN_ALL_FOLDS = False # True = run every fold and report mean +/- std (5x the time)
EPOCHS = 15
BATCH_SIZE = 16
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.05
VAL_FRAC = 0.15       # fraction of the *training players* held out for validation
WORKERS = 4
SEED = 0

# X3D-M input settings (from PyTorchVideo)
NUM_FRAMES, CROP, SHORT_SIDE = 16, 224, 256

## 2. Download only `VIDEO_RGB` from GitHub (sparse checkout, ~3.7 GB)

In [3]:
import subprocess

REPO_URL = "https://github.com/THETIS-dataset/dataset.git"

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if RGB_ROOT.is_dir() and any(RGB_ROOT.rglob("*.avi")):
    print(f"{RGB_ROOT} already has videos, skipping download.")
else:
    run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1", REPO_URL, str(DATA_DIR)])
    run(["git", "-C", str(DATA_DIR), "sparse-checkout", "init", "--no-cone"])
    run(["git", "-C", str(DATA_DIR), "sparse-checkout", "set", "VIDEO_RGB/"])
    branch = subprocess.run(["git", "-C", str(DATA_DIR), "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
    run(["git", "-C", str(DATA_DIR), "checkout", branch])

print("videos found:", sum(1 for _ in RGB_ROOT.rglob("*.avi")))

$ git clone --filter=blob:none --no-checkout --depth 1 https://github.com/THETIS-dataset/dataset.git /kaggle/working/thetis


Cloning into '/kaggle/working/thetis'...


$ git -C /kaggle/working/thetis sparse-checkout init --no-cone
$ git -C /kaggle/working/thetis sparse-checkout set VIDEO_RGB/
$ git -C /kaggle/working/thetis checkout main
Your branch is up to date with 'origin/main'.
videos found: 1980


Already on 'main'


## 3. Build and validate the manifest

Player ids are parsed from filenames (e.g. `p12`). Per the dataset README, p1-p31 are beginners and p32-p55 are experts.
**Check the warnings below**: if player ids can't be parsed, the player-grouped split can't work.

In [4]:
import re
import numpy as np
import pandas as pd

VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}
PLAYER_RE = re.compile(r"(?<![a-z])p(\d+)(?!\d)", re.IGNORECASE)
LAST_BEGINNER = 31

def build_manifest(root: Path) -> pd.DataFrame:
    rows, unmatched = [], []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in VIDEO_EXTS:
            continue
        m = PLAYER_RE.search(p.stem)
        if not m:
            unmatched.append(p.name)
            continue
        player = int(m.group(1))
        rows.append({"path": str(p.relative_to(root)), "class": p.parent.name, "player": player,
                     "level": "beginner" if player <= LAST_BEGINNER else "expert"})
    if unmatched:
        print(f"WARNING: {len(unmatched)} files had no player id, e.g. {unmatched[:3]}")
    return pd.DataFrame(rows)

df = build_manifest(RGB_ROOT)
assert len(df) > 0, "No videos with a parsable player id. Fix PLAYER_RE for your filenames."

print(f"videos: {len(df)} | classes: {df['class'].nunique()} | players: {df['player'].nunique()}")
for name, got, want in [("videos", len(df), 1980), ("classes", df["class"].nunique(), 12),
                        ("players", df["player"].nunique(), 55)]:
    if got != want:
        print(f"WARNING: expected {want} {name}, found {got}")
print(df.groupby("class").size().to_string())
df.head()

videos: 1980 | classes: 12 | players: 55
class
backhand               165
backhand2hands         165
backhand_slice         165
backhand_volley        165
flat_service           165
forehand_flat          165
forehand_openstands    165
forehand_slice         165
forehand_volley        165
kick_service           165
slice_service          165
smash                  165


,path,class,player,level
0,backhand/p10_backhand_s1.avi,backhand,10,beginner
1,backhand/p10_backhand_s2.avi,backhand,10,beginner
2,backhand/p10_backhand_s3.avi,backhand,10,beginner
3,backhand/p11_backhand_s1.avi,backhand,11,beginner
4,backhand/p11_backhand_s2.avi,backhand,11,beginner


## 4. (Optional) Pre-resize videos

In [5]:
import cv2
from multiprocessing import Pool

def _resize_one(job):
    src, dst, short = job
    if dst.exists():
        return True
    cap = cv2.VideoCapture(str(src))
    fps = cap.get(cv2.CAP_PROP_FPS) or 18.0
    writer, n = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if writer is None:
            h, w = frame.shape[:2]
            s = short / min(h, w)
            size = (round(w * s), round(h * s))
            dst.parent.mkdir(parents=True, exist_ok=True)
            writer = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"MJPG"), fps, size)
        writer.write(cv2.resize(frame, size, interpolation=cv2.INTER_AREA))
        n += 1
    cap.release()
    if writer is not None:
        writer.release()
    return n > 0

if RESIZE:
    jobs = [(RGB_ROOT / r, RESIZED_ROOT / Path(r), SHORT_SIDE) for r in df["path"]]
    with Pool(WORKERS) as pool:
        ok = list(pool.imap_unordered(_resize_one, jobs))
    print(f"resized {sum(ok)}/{len(jobs)} videos -> {RESIZED_ROOT}")
    DATA_ROOT = RESIZED_ROOT
else:
    DATA_ROOT = RGB_ROOT
print("training will read from:", DATA_ROOT)

training will read from: /kaggle/working/thetis/VIDEO_RGB


## 5. Dataset, splits and model

In [6]:
import random, time
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
STD = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)

classes = sorted(df["class"].unique())
cls_to_idx = {c: i for i, c in enumerate(classes)}
paths = np.array([str(DATA_ROOT / p) for p in df["path"]])
labels = df["class"].map(cls_to_idx).values
groups = df["player"].values


def read_frames(path):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        s = SHORT_SIDE / min(h, w)
        if abs(s - 1.0) > 1e-3:
            frame = cv2.resize(frame, (round(w * s), round(h * s)), interpolation=cv2.INTER_LINEAR)
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames decoded from {path}")
    return frames


def sample_indices(n, train):
    """One frame per temporal segment: random in train, middle in eval."""
    edges = np.linspace(0, n, NUM_FRAMES + 1)
    idx = []
    for i in range(NUM_FRAMES):
        lo = int(edges[i]); hi = max(lo + 1, int(edges[i + 1]))
        idx.append(random.randrange(lo, hi) if train else (lo + hi - 1) // 2)
    return np.clip(idx, 0, n - 1)


class ThetisDataset(Dataset):
    def __init__(self, paths, labels, train):
        self.paths, self.labels, self.train = paths, labels, train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        frames = read_frames(self.paths[i])
        idx = sample_indices(len(frames), self.train)
        clip = torch.from_numpy(np.stack([frames[j] for j in idx]))      # T,H,W,C uint8
        clip = clip.permute(3, 0, 1, 2).float() / 255.0                  # C,T,H,W
        _, _, H, W = clip.shape
        if self.train:
            top, left = random.randint(0, H - CROP), random.randint(0, W - CROP)
            clip = clip[:, :, top:top + CROP, left:left + CROP]
            clip = (clip * random.uniform(0.8, 1.2) + random.uniform(-0.1, 0.1)).clamp(0, 1)
        else:
            top, left = (H - CROP) // 2, (W - CROP) // 2
            clip = clip[:, :, top:top + CROP, left:left + CROP]
        return (clip - MEAN) / STD, int(self.labels[i])


def make_splits(fold, n_folds=N_FOLDS, val_frac=VAL_FRAC, seed=SEED):
    """Player-grouped, class-stratified test fold; then a player-grouped val split from train."""
    idx = np.arange(len(labels))
    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    train_val, test = list(skf.split(idx, labels, groups))[fold]
    gss = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    tr, va = next(gss.split(train_val, labels[train_val], groups[train_val]))
    train, val = train_val[tr], train_val[va]
    assert not set(groups[train]) & set(groups[test]), "player leakage train/test"
    assert not set(groups[val]) & set(groups[test]), "player leakage val/test"
    assert not set(groups[train]) & set(groups[val]), "player leakage train/val"
    return train, val, test


def build_model(num_classes):
    model = torch.hub.load("facebookresearch/pytorchvideo", "x3d_m", pretrained=True, trust_repo=True)
    head = model.blocks[5]
    head.proj = nn.Linear(head.proj.in_features, num_classes)
    return model


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            out = model(x)
            loss = criterion(out, y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total, loss_sum / total

## 6. Training loop

In [7]:
def train_fold(fold, epochs=EPOCHS):
    random.seed(SEED + fold); np.random.seed(SEED + fold); torch.manual_seed(SEED + fold)
    tr, va, te = make_splits(fold)
    print(f"[fold {fold}] train {len(tr)} | val {len(va)} | test {len(te)}  (split by player)")

    def loader(ix, train):
        ds = ThetisDataset(paths[ix].tolist(), labels[ix], train)
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train,
                          num_workers=WORKERS, pin_memory=device.type == "cuda",
                          persistent_workers=WORKERS > 0)

    train_loader, val_loader, test_loader = loader(tr, True), loader(va, False), loader(te, False)

    model = build_model(len(classes)).to(device)
    head_params = list(model.blocks[5].proj.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.AdamW(
        [{"params": backbone_params, "lr": LR_BACKBONE}, {"params": head_params, "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    ckpt_path = WORK / f"x3d_m_thetis_fold{fold}.pt"
    history, best_val, best_epoch = [], 0.0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0, run_loss, n = time.time(), 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * y.size(0)
            n += y.size(0)
        scheduler.step()
        val_acc, val_loss = evaluate(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": run_loss / n, "val_loss": val_loss, "val_acc": val_acc})
        print(f"epoch {epoch:02d}/{epochs} | train loss {run_loss / n:.3f} | "
              f"val loss {val_loss:.3f} acc {val_acc:.3f} | {time.time() - t0:.0f}s")
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            torch.save({"model": model.state_dict(), "classes": classes, "epoch": epoch}, ckpt_path)

    # Test once, at the epoch chosen on validation
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["model"])
    test_acc, test_loss = evaluate(model, test_loader, criterion)
    print(f"[fold {fold}] best val acc {best_val:.3f} (epoch {best_epoch}) -> TEST acc {test_acc:.3f}\n")
    return {"fold": fold, "best_val": best_val, "best_epoch": best_epoch,
            "test_acc": test_acc, "history": history, "ckpt": str(ckpt_path)}

## 7. Run

In [ ]:
folds = range(N_FOLDS) if RUN_ALL_FOLDS else [FOLD]
results = [train_fold(f) for f in folds]

accs = np.array([r["test_acc"] for r in results])
print("test acc per fold:", np.round(accs, 3))
print(f"mean {accs.mean():.3f}" + (f" +/- {accs.std():.3f}" if len(accs) > 1 else ""))

[fold 0] train 1332 | val 252 | test 396  (split by player)
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/X3D_M.pyth" to /root/.cache/torch/hub/checkpoints/X3D_M.pyth


100%|██████████| 29.4M/29.4M [00:00<00:00, 118MB/s] 


epoch 01/15 | train loss 2.002 | val loss 1.604 acc 0.464 | 113s
epoch 02/15 | train loss 1.376 | val loss 1.404 acc 0.575 | 110s
epoch 03/15 | train loss 1.181 | val loss 1.379 acc 0.639 | 111s
epoch 04/15 | train loss 1.056 | val loss 1.380 acc 0.627 | 111s
epoch 05/15 | train loss 0.955 | val loss 1.378 acc 0.639 | 110s
epoch 06/15 | train loss 0.896 | val loss 1.456 acc 0.651 | 111s
epoch 07/15 | train loss 0.848 | val loss 1.345 acc 0.655 | 112s
epoch 08/15 | train loss 0.807 | val loss 1.409 acc 0.659 | 112s
epoch 09/15 | train loss 0.762 | val loss 1.365 acc 0.683 | 111s
epoch 10/15 | train loss 0.757 | val loss 1.382 acc 0.671 | 112s
epoch 11/15 | train loss 0.720 | val loss 1.366 acc 0.690 | 111s
epoch 12/15 | train loss 0.698 | val loss 1.382 acc 0.683 | 113s


## 8. Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for r in results:
    h = pd.DataFrame(r["history"])
    ax[0].plot(h["epoch"], h["train_loss"], label=f"train f{r['fold']}")
    ax[0].plot(h["epoch"], h["val_loss"], "--", label=f"val f{r['fold']}")
    ax[1].plot(h["epoch"], h["val_acc"], label=f"fold {r['fold']}")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)
ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 9. (Optional) Per-class test report for the last fold

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

fold = results[-1]["fold"]
_, _, te = make_splits(fold)
test_loader = DataLoader(ThetisDataset(paths[te].tolist(), labels[te], False),
                         batch_size=BATCH_SIZE, num_workers=WORKERS)
model = build_model(len(classes)).to(device)
model.load_state_dict(torch.load(results[-1]["ckpt"], map_location=device)["model"])
model.eval()

preds, ys = [], []
with torch.no_grad():
    for x, y in test_loader:
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            preds += model(x.to(device)).argmax(1).cpu().tolist()
        ys += y.tolist()

print(classification_report(ys, preds, target_names=classes, digits=3))
print(pd.DataFrame(confusion_matrix(ys, preds), index=classes, columns=range(len(classes))))